## Optimizing for divergence between correlation and causation
In this notebook, we will directly optimize for disagreement between functional similarity and correlative similarity.

# 1. Define the Models

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
from vis_training import get_model_and_processor

overwrite = False # if True, will not load from saved values

models = []
processors = []

model_names = [
    "microsoft/resnet-18",
    "microsoft/resnet-18",
]
for model_name in model_names:
    # Models are defined as the backbone (frozen parameters, no head)
    # with an untrained head (unfrozen parameters) that will be trained
    # on the cifar10 dataset before training the MAS alignment
    model, proc = get_model_and_processor(model_name)
    models.append(model)
    processors.append(proc)


# 2. Load the input data (and labels) and finetune

In [3]:
import torch
from vis_training import train_model, get_dataloaders
from torchvision import datasets
import os

batch_size = 128
val_batch_size = 1000
num_workers = 1
num_epochs = 10
lr = 0.001
model_save_dir = "/data2/grantsrb/vision_mas/models"
data_root = "/data2/grantsrb/cifar10"

train_ds = datasets.CIFAR10(root=data_root, train=True, download=True)
test_ds = datasets.CIFAR10(root=data_root, train=False, download=True)

train_loaders = []
test_loaders = []
for i, (model, processor) in enumerate(zip(models, processors)):
    train_loader, test_loader = get_dataloaders(
        train_dataset=train_ds,
        test_dataset=test_ds,
        processor=processor,
        batch_size=batch_size,
        val_batch_size=val_batch_size,
        num_workers=num_workers,
    )
    train_loaders.append(train_loader)
    test_loaders.append(test_loader)


In [ ]:
for i, (model, processor) in enumerate(zip(models, processors)):
    train_loader = train_loaders[i]
    test_loader = test_loaders[i]
    model_name = model_names[i].replace("/", "_")
    model_save_path = f"{model_save_dir}/{model_name}_cifar10_sd_{i}.pt"
    if os.path.exists(model_save_path) and not overwrite:
        print(f"Loading model from {model_save_path}")
        model.load_state_dict(torch.load(model_save_path))
    else:
        print(f"Finetuning model {i}")
        try:
            model = train_model(
                model=model,
                train_loader=train_loader,
                test_loader=test_loader,
                hyperparameters={"lr": lr, "num_epochs": num_epochs},
            )
        except KeyboardInterrupt:
            print("Interrupted training, continuing...")
            pass
    
    # From here on, we will update more than just the classification head.
    for p in model.parameters():
        p.requires_grad = True
    
    torch.save(model.state_dict(), model_save_path)


Loading model from /data2/grantsrb/vision_mas/models/microsoft_resnet-18_cifar10_sd_0.pt
Loading model from /data2/grantsrb/vision_mas/models/microsoft_resnet-18_cifar10_sd_1.pt


# 3. Optimize for low CKA (maintaining performance)

In [5]:
import torch

layer_names = [ # need a layer name for each model
    "backbone.encoder.stages.1.layers.0",
    "backbone.encoder.stages.1.layers.0",
]

device = 0 if torch.cuda.is_available() else "cpu"

### Collect training and validation sets

In [ ]:
from vis_training import get_input_outputs, get_dataloader

train_sets = [] # will store (preprocessed input, model output) tuples for each model
valid_sets = []
for i, (model, processor, val_loader) in enumerate(zip(models, processors, test_loaders)):
    train_loader = get_dataloader(
        dataset=train_ds,
        processor=processor,
        batch_size=batch_size,
        num_workers=num_workers,
        shuffle=False,
    )

    train_save_path = f"{model_save_dir}/cka_train_sets_{i}.pt"
    valid_save_path = f"{model_save_dir}/cka_valid_sets_{i}.pt"
    if os.path.exists(train_save_path) and not overwrite:
        print(f"Loading training set from {train_save_path}")
        train_set = torch.load(train_save_path)
        valid_set = torch.load(valid_save_path)
    else:
        model.eval()
        model.to(device)
        print(f"Collecting training set {i}")
        with torch.no_grad():
            train_set = get_input_outputs(
                model=model,
                data_loader=train_loader,
                verbose=True,
            )
            train_sets.append(train_set)
            valid_set = get_input_outputs(
                model=model,
                data_loader=val_loader,
                verbose=True,
            )
            valid_sets.append(valid_set)
        torch.save(train_set, train_save_path)
        torch.save(valid_set, valid_save_path)
    model.cpu()

100%|██████████| 10/10 [00:15<00:00,  1.60s/it]


100%|██████████| 10/10 [00:15<00:00,  1.54s/it]


In [ ]:
import torch.optim as optim
from vis_training import minimize_cka_one_epoch
from vis_similarity import get_cka
from hooks import hook_model_layer
import numpy as np

one_hot_loss = False
batch_size = 128
num_epochs = 10

cka_dict = dict()
for src_idx in range(len(models)):
    src_handle, src_comms_dict = hook_model_layer(
        models[src_idx], layer_names[src_idx], key="actvs"
    )
    for trg_idx in range(src_idx+1, len(models)): # CKA is symmetric
        if src_idx == trg_idx:
            continue
        print("Starting Models", src_idx, trg_idx)
        trg_handle, trg_comms_dict = hook_model_layer(
            models[trg_idx], layer_names[trg_idx], key="actvs"
        )
        model_pair = (models[src_idx].to(device), models[trg_idx].to(device))
        optimizer = optim.RMSprop(
            [p for p in model_pair[0].parameters()]
            + [p for p in model_pair[1].parameters()],
            lr=lr
        )

        train_ckas = []
        train_accs = []
        train_losses = []
        valid_ckas = []
        valid_accs = []
        valid_losses = []
        for epoch in range(num_epochs):
            print(f"Epoch {epoch}")
            models = [model.train() for model in model_pair]
            train_accs, train_losses, train_cka = minimize_cka_one_epoch(
                models=model_pair,
                data_sets=[train_sets[src_idx], train_sets[trg_idx]],
                optimizer=optimizer,
                batch_size=batch_size,
                comms_dicts=[src_comms_dict, trg_comms_dict],
                one_hot_loss=one_hot_loss,
            )
            train_ckas.append(torch.mean(train_cka).item())
            train_accs.append(
                [torch.mean(train_acc).item() for train_acc in train_accs]
            )
            train_losses.append(
                [torch.mean(train_loss).item() for train_loss in train_losses]
            )

            models = [model.eval() for model in model_pair]
            valid_accs, valid_losses, valid_cka = minimize_cka_one_epoch(
                models=model_pair,
                data_sets=[valid_sets[src_idx], valid_sets[trg_idx]],
                batch_size=batch_size,
                comms_dicts=[src_comms_dict, trg_comms_dict],
                one_hot_loss=one_hot_loss,
            )
            valid_ckas.append(torch.mean(valid_cka).item())
            valid_accs.append(
                [torch.mean(valid_acc).item() for valid_acc in valid_accs]
            )
            valid_losses.append(
                [torch.mean(valid_loss).item() for valid_loss in valid_losses]
            )
            print()
            print("Model 0")
            print(f"\tTrain -- Acc: {train_accs[-1]}, Loss: {train_losses[-1]}")
            print(f"\tValid -- Acc: {valid_accs[-1]}, Loss: {valid_losses[-1]}")
            print("Model 1")
            print(f"\tTrain -- Acc: {train_accs[-1]}, Loss: {train_losses[-1]}")
            print(f"\tValid -- Acc: {valid_accs[-1]}, Loss: {valid_losses[-1]}")
            print(f"\tCKA -- Train {train_ckas[-1]}, Valid: {valid_ckas[-1]}")

        cka_dict[(src_idx, trg_idx)] = valid_ckas[-1]
        trg_handle.remove()
    src_handle.remove()



Epoch 0


  0%|          | 0/1 [00:00<?, ?it/s]

In [6]:
torch.save(actvs_train_sets, f"{model_save_dir}/actvs_train_sets.pt")
torch.save(actvs_valid_sets, f"{model_save_dir}/actvs_valid_sets.pt")

In [8]:
actvs_train_sets = torch.load(f"{model_save_dir}/actvs_train_sets.pt")
actvs_valid_sets = torch.load(f"{model_save_dir}/actvs_valid_sets.pt")

# 4. Instantiate the MAS alignment object

In [24]:
og_models = [model for model in models]
og_processors = [processor for processor in processors]
og_layer_names = [layer_name for layer_name in layer_names]
og_actvs_train_sets = [actvs_train_set for actvs_train_set in actvs_train_sets]
og_actvs_valid_sets = [actvs_valid_set for actvs_valid_set in actvs_valid_sets]

In [29]:
from alignment import MASAlignment
from hooks import hook_vision_model

subspace_size = 128 # number of dimensions used in the patching
mtx_type = "orthogonal"
normalize = False # learns to normalize the activations before the transformation matrix
batch_norm = True
identity_rot = False # uses the identity rotation matrix (only use for debugging)
debug = True

if debug:
    models = [og_models[0] for _ in models]
    processors = [og_processors[0] for _ in processors]
    layer_names = [og_layer_names[0] for _ in layer_names]
    actvs_train_sets = [og_actvs_train_sets[0] for _ in actvs_train_sets]
    actvs_valid_sets = [og_actvs_valid_sets[0] for _ in actvs_valid_sets]
else:
    models = [model for model in og_models]
    processors = [processor for processor in og_processors]
    layer_names = [layer_name for layer_name in og_layer_names]
    actvs_train_sets = [actvs_train_set for actvs_train_set in og_actvs_train_sets]
    actvs_valid_sets = [actvs_valid_set for actvs_valid_set in og_actvs_valid_sets]

# the alignment object contains the transformation matrices for each model
# and the patching masks for each model. It also contains the logic for
# performing an alignment intervention (i.e. transferring causal activity
# from a source model to a target model).
alignment = MASAlignment(
    model_dims=[data["actvs"].shape[1] for data in actvs_train_sets],
    mtx_type=mtx_type,
    subspace_sizes=subspace_size,
    normalize=normalize,
    batch_norm=batch_norm,
    identity_rot=identity_rot,
)

# We need to hook the models in order to perform the patching intervention
# at the desired layer.
try:
    hooks = [hook.remove() for hook in hooks]
except:
    pass

hooks = []
for model,layer in zip(models, layer_names):
    hooks.append(
        hook_vision_model(model, layer, alignment)
    )

Using subspace sizes: [128]


# 5. Train the MAS alignment

In [ ]:
from vis_utils import train_mas_alignment_one_epoch, evaluate_mas_alignment
import torch.optim as optim
import numpy as np
import pandas as pd

num_epochs = 25
batch_size = 256
lr = 0.0005
one_hot_loss = False
label_smoothing = 0.0
train_directions = None
ground_truth_labels = False
val_batch_size = 1000

device = 0 if torch.cuda.is_available() else "cpu"
alignment.to(device)
alignment.train()
optimizer = optim.RMSprop(alignment.parameters(), lr=lr)
train_dfs = []
valid_dfs = []
for epoch in range(num_epochs):
    print(f"Epoch {epoch} - Training")
    train_df = train_mas_alignment_one_epoch(
        models=models,
        alignment=alignment,
        actvs_sets=actvs_train_sets,
        batch_size=batch_size,
        optimizer=optimizer,
        one_hot_loss=one_hot_loss,
        label_smoothing=label_smoothing,
        train_directions=train_directions,
        use_ground_truth_labels=ground_truth_labels,
    )
    print(f"Epoch {epoch} - Validating")
    valid_df = evaluate_mas_alignment(
        models=models,
        alignment=alignment,
        actvs_sets=actvs_valid_sets,
        batch_size=val_batch_size,
        one_hot_loss=one_hot_loss,
        use_ground_truth_labels=ground_truth_labels,
    )

    cols = ["actn_loss","cl_loss","acc"]
    groups = ["src_idx","trg_idx"]

    train = train_df.groupby(groups)[cols].mean().reset_index()
    train["epoch"] = epoch
    train_dfs.append(train)

    valid = valid_df.groupby(groups)[cols].mean().reset_index()
    valid["epoch"] = epoch
    valid_dfs.append(valid)
    print()
    print(train.sort_values(by=groups,ascending=True))
    valid.columns = ["valid_"+col if col in cols else col for col in valid.columns]
    print(valid.sort_values(by=groups,ascending=True))
    print(
        "Train IIA:", round(np.min(train["acc"]), 5),
        "|| Loss:", round(np.max(train["actn_loss"]), 5)
    )
    print(
        "Valid IIA:", round(np.min(valid["valid_acc"]), 5),
        "|| Loss:", round(np.max(valid["valid_actn_loss"]), 5)
    )
train_df = pd.concat(train_dfs)
valid_df = pd.concat(valid_dfs)


Epoch 0
Batch 9984/10000 IIA: 0.9375 Loss: 0.6057357788085938                                                               
   src_idx  trg_idx  actn_loss  cl_loss       acc  epoch
0        0        0   0.566333      0.0  0.999841      0
1        0        1   0.688486      0.0  0.874251      0
2        1        0   0.689693      0.0  0.872684      0
3        1        1   0.566333      0.0  0.999920      0
   src_idx  trg_idx  valid_actn_loss  valid_cl_loss  valid_acc  epoch
0        0        0         0.572126            0.0   1.000000      0
1        0        1         0.694260            0.0   0.873145      0
2        1        0         0.692942            0.0   0.872363      0
3        1        1         0.572126            0.0   1.000000      0
Train IIA: 0.87268 || Loss: 0.68969
Valid IIA: 0.87236 || Loss: 0.69426
Epoch 1
Batch 9984/10000 IIA: 0.9375 Loss: 0.6039549112319946                                                               
   src_idx  trg_idx  actn_loss  cl_loss    

In [ ]:
import datetime

def get_timestamp():
    return datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

In [ ]:
train_df.columns = ["train_"+col if col in cols else col for col in train_df.columns]
valid_df.columns = ["valid_"+col if col in cols else col for col in valid_df.columns]
main_df = pd.merge(train_df, valid_df, on=groups+["epoch"])

timestamp = get_timestamp()
m1 = model_names[0].replace("/", "_")
m2 = model_names[1].replace("/", "_")
main_df.to_csv(f"{m1}_{m2}_mas_results_{timestamp}.csv", index=False, header=True)
main_df.head()

# 6. Evaluate results

In [ ]:
main_df["direction"] = main_df["src_idx"].astype(str) + "->" + main_df["trg_idx"].astype(str)
cols = ["train_acc","valid_acc", "train_actn_loss", "valid_actn_loss"]
all_dfs = [main_df]
for agg in ["min","max","mean"]:
    df = main_df.groupby(["epoch"])[cols].agg(agg).reset_index()
    df["epoch"] = df.index
    df["direction"] = agg
    all_dfs.append(df)
    break
#all_dfs = pd.concat(all_dfs)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig = plt.figure(figsize=(7,7))
ax = plt.gca()

sns.lineplot(data=main_df, x="epoch", y="train_acc", hue="src_idx", ax=ax)